## Práctica de  clasificación con varios datasets y test estadísticos

Nombre: Danny Carrera 
Máster en Ingeniería Informática

In [1]:
# %load ../../standard_import.txt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from test_helper import Test

pd.set_option('display.notebook_repr_html', False)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_seq_items', None)
 
#%config InlineBackend.figure_formats = {'pdf',}
%matplotlib inline  

Esta práctica consiste en aprender a utilizar los tests estadísticos para realizar el análisis de los resultados obtenidos por varios clasificadores en un entorno experimental compuesto por varios datasets. En concreto se deben aprender a utilizar:
* El test de Shapiro-Wilk para evaluar la normalidad de los datos.
* El test de Bartlett para evaluar la homocedasticidad de los datos.
* Los test paramétricos o no paramétricos adecuados para realizar el test de la hipótesis.

Para ello, lo primero que se debe hacer es descomprimir el archivo ```datasets.zip```, en el que tenéis los 13 datasets que utilizaremos en la práctica, en una carpeta llamada **datasets**. A continuación se debe obtener una lista con los nombres de los 13 ficheros de la carpeta descomprimida.

Para ello vamos a utilizar métodos facilitados en dos librerías: *os* y *os.path*. En concreto vamos a utilizar los siguientes tres métodos:
* *listdir*: función perteneciente a *os* que nos provee un listado del contenido de un directorio. https://www.tutorialspoint.com/python/os_listdir.htm
* *isfile*: función perteneciente a *os.path* que determina si un path es  un fichero o no. 
* *join*: función perteneciente a *os.path* que compone un path a partir de varios paths (strings). 

Estos dos últimos métodos se pueden consultar en https://docs.python.org/2/library/os.path.html.

Cread una función llamada *ls* que reciba como parámetros de entrada un path y una extensión y devuelva una lista con el nombre de todos los archivos del path que tengan la extensión deseada. Para saber si un string acaba con un determinado substring podéis utilizar el método *endswith* de la clase *string*.

Utilizad dicha función para obtener los archivos con extesión '.dat' de la carpeta *datasets* y almacenadla en una variable llamada *datasets*.

In [2]:
# Se importan las librerías necesarias
from os import listdir
from os.path import isfile, join

# Se crea la función ls
def ls(path, extension):
    return [f for f in listdir(path) if isfile(join(path, f)) and f.endswith(extension)]

# Se obtiene el listado de datasets
datasets = ls("datasets", ".dat") #.dat porque busca una carpeta llamada datasets en la misma carpeta donde estoy trabajando 
datasets

['appendicitis.dat',
 'balance.dat',
 'bupa.dat',
 'ecoli.dat',
 'glass.dat',
 'haberman.dat',
 'iris.dat',
 'newthyroid.dat',
 'page-blocks.dat',
 'pima.dat',
 'wine.dat',
 'wisconsin.dat',
 'yeast.dat']

Para leer el contenido de los ficheros con los datos de cada dataset se os ha facilitado la clase dataset, archivo **dataset.py**. Esta clase permite realizar la lectura de datos de problemas de clasificación que estén en formato KEEL. La clase dataset tiene un método llamado lecturaDatos:
* Parámetro de entrada: nombre del archivo que contiene los datos del problema a resolver
* Parámetros de salida: un objeto dataset que tiene varios campos con diferente información (se acceden con nombreVariable.nombreCampo):
    * data: contiene los datos leídos del fichero. Los atributos categóricos han sido transformados utilizando la codificación ordinal.
    * target: contiene las clases del problema (también codificadas mediante representación ordinal).
    * feature_names: es una lista con los nombres de las variables de entrada del problema.
    * target_names: es una lista con los nombres de las clases del problema.
    * categorical: es una lista de booleanos y cada valor especifica si el atributo correspondiente es categórico o no.

En esta práctica debéis:
* Utilizar el clasificador de los k vecinos más cercanos utilizando los valores 1, 3, 5 y 7 para el parámetro k. 
* Realizar una validación cruzada de 5 particiones con la semilla 123. 
* Recordad que para obtener el rendimiento del clasificador podéis utilizar el método *accuracy_score* de la librería *metrics*. 

Por último, en esta práctica las variables de entrada de los problemas de clasificación tienen rangos diferentes por lo que hay que normalizarlos para que todos estén en el mismo rango. Para ello debéis utilizar la clase [**StandardScaler**](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) de la librería *sklearn.preprocessing*. 

Para ello, vamos a establecer una secuencia de técnicas a aplicar a los datos. La clase de Python que vamos a aplicar permite establecer todas las técnicas de pre-procesamiento a aplicar (y en el orden deseado) y finalizar con una técnica de predicción. Una vez que se ha establecido la secuencia, al realizar el aprendizaje, se aplicará el aprendizaje de cada componente de forma secuencial y por tanto, los componentes aprenderán sobre el resultado dado por los componentes previos de la secuencia (lo mismo pasa al realizar la predicción). De este modo, ahorramos código y minimizamos las opciones de cometer fallos de programación y incurrir en *data leakage* que afecten al resultado obtenido. 

La clase que nos ofrece esta posibilidad se llama [**Pipeline**](http://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline) y está dentro de la librería *pipeline* de Scikit-learn. La llamada al constructor de esta clase consiste en un conjunto de tuplas del tipo (nombreFase, objeto), cuyo significado es:
* nombreFase: string que establece el nombre de la fase. Por ejemplo 'estandarizacion' o 'clasificador'
* objeto: variable en la que se almacena la llamada al constructor de lo que se desee hacer. Por ejemplo StandardScaler() o neighbors.KNeighborsClassifier()

Es decir, si quisiéramos combinar los procesos mencionados anteriormente deberíamos realizar la siguiente llamada:

    pipeline = Pipeline([('estandarizacion', StandardScaler()),  ('clasificador', neighbors.KNeighborsClassifier())])
    
Hay que destacar que los objetos de todas las fases de la secuencia (excepto de la última) tienen que tener los métodos *fit* y *transform*, para que puedan aprender de los datos y transformarlos en consecuencia. El último objeto debe tener el método *fit*, para aprender de los datos, y el método *predict* para poder realizar nuevas predicciones. Es decir, el último objeto debe ser un modelo de clasificación o regresión.

El objeto combinado, la Pipeline generada, dispone de los siguientes métodos:
* fit: Recibe como parámetros de entrada los datos de entrada (X) y de salida (Y). Cada objeto de cada fase aprende en base a dichos datos.
* predict: Recibe como parámetro de entrada los datos de entrada (X). Realiza la predicción realizando lo siguiente:
    * Primero se aplican las transformaciones de datos por medio de los primeros objetos (llaman a sus respectivas funciones transform)
    * Finalmente, se aplica al objeto de la última fase (clasificador o modelo de regresión) para realizar la predicción correspondiente a los datos de entrada (llamada a su método predict). 

Entrenad la Pipeline con los datos de entrenamiento y aplicadla con los datos de validación (test). Finalmente, guardamos los resultados en un archivo .csv que será leído para realizar la comparación de clasificadores mediante el estudio estadístico apropiado.

In [3]:
# Se importan las librerías necesarias
from dataset import lecturaDatos
import numpy as np
import pandas as pd
from sklearn import neighbors
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Lista con los valores de k a utilizar
K = [1, 3, 5, 7]

# Matriz para almacenar los resultados
resultadosEjecucion = np.zeros((len(datasets), len(K)))

# Bucle para recorrer los diferentes valores de k
for i, k in enumerate(K):

    # Lista vacía para almacenar los resultados en cada dataset
    accDatasets = []

    # Bucle para recorrer los datasets
    for fichero in datasets:

        # Se leen los datos del dataset
        d = lecturaDatos(join("datasets", fichero))

        # Se crea el objeto de validación cruzada
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)

        # Lista vacía para almacenar el rendimiento en cada partición
        accParticiones = []

        # Bucle para realizar la validación cruzada
        for train_index, test_index in cv.split(d.data, d.target):

            # Se crea el objeto de Pipeline
            pipeline = Pipeline([
                ('estandarizacion', StandardScaler()),
                ('clasificador', neighbors.KNeighborsClassifier(n_neighbors=k))
            ])

            # Se entrena la Pipeline
            pipeline.fit(d.data[train_index], d.target[train_index])

            # Se predicen las clases de los ejemplos de test
            predicciones = pipeline.predict(d.data[test_index])

            # Se obtiene el rendimiento de test
            acc = accuracy_score(d.target[test_index], predicciones) * 100

            # Se añade el rendimiento a la lista
            accParticiones.append(acc)

        # Rendimiento medio de las 5 particiones
        accDatasets.append(np.mean(accParticiones))

    # Se almacenan los resultados
    resultadosEjecucion[:, i] = accDatasets


# Se crea el DataFrame
dfKNN = pd.DataFrame(
    resultadosEjecucion,
    index=datasets,
    columns=['k1', 'k3', 'k5', 'k7']
)

# Nombre del índice
dfKNN.index.name = 'Metodo'

# Se guarda el resultado en CSV
dfKNN.to_csv('resultadosKNN.csv')

In [8]:
Test.assertEquals(list(map(lambda x: round(x, 2), resultadosEjecucion.mean(axis=0))), [79.87, 81.16, 81.48, 81.47], 'Resultados incorrectos')

1 test passed.


Al igual que en la práctica anterior, vamos a utilizar la librería [*StaTDS*](https://statds.readthedocs.io/en/latest/) para realizar el estudio estadístico, ya que nos facilita todos los test necesarios para realizarlo. Al pinchar sobre cada método podemos ver la información detallada sobre los argumentos de entrada y de salida de cada test. Todos los test utilizados en esta práctica:
* Devuleven el p-valor y un string *hypothesis* que contiene la información sobre si se rechaza la hipóstesis nula o no.
* En el caso de los test post-hoc se puede mostrar visualmente el resultado haciendo show sobre el segundo argumento que devuelve.

Todos los tests (tanto los de evaluación de la normalidad, de la homocedasticidad o los de test de hipótesis) devuelven un p-valor. En caso de que ese valor sea menor que el valor de significancia ($\alpha$) determinado de antemano, se rechaza la hipótesis nula. Esta hipótesis, en el caso de:
* Evaluar la normalidad es que los datos siguen una distribución normal
* Evaluar la homocedasticidad es que los datos tienen varianzas homogéneas.

Por tanto, si se rechazan dichas hipótesis significa que los datos no siguen una distribución normal o no tienen varianzas homogéneas, respectivamente.

La hipóstesis nula a la hora de realizar la comparación de clasificadores es la misma que la explicada en la teoría, es decir, que los clasificadores no presentan diferencias evidentes en su rendimiento.

Importamos todos los test que nos van a ser necesarios para desarrollar la práctica.

In [9]:
from statds.normality import shapiro_wilk_normality
from statds.homoscedasticity import bartlett_test
from statds.no_parametrics import friedman_aligned_ranks, holm, wilcoxon
from statds.parametrics import anova_cases, t_test_unpaired

A continuación se debe realizar el estudio estadístico pertinente para determinar si alguna configuración de KNN es mejor que otra. Para ello, se deben seguir los siguientes pasos:
* Realizar los test de normalidad (Shapiro-Wilk: *shapiro_wilk_normality*) a los resultados de cada clasificador (a cada columna del DataFrame).
* Realizar el test de homocedasticidad (Bartlett: *bartlett_test*)
* Aplicar el estudio estadístico pertinente
    * Si los datos cumplen ambas condiciones aplicar test estadísticos paramétricos
        * comparación por pares si son dos clasificadores (t-test: *t_test_unpaired*)
        * comparación múltiple si son más de dos clasificadores (ANOVA: *anova_cases*)
    * Si los datos no cumplen ambas condiciones aplicar test estadísticos no paramétricos
        * comparación por pares si son dos clasificadores (Wilcoxon: *wilcoxon*)
        * comparación múltiple si son más de dos clasificadores (rangos alineados de Friedman: *friedman_aligned_ranks*)
* Si el test de comparación múltiple anterior rechaza la hipótesis nula, realizad el test post-hoc de Holm: *holm* (solo en el caso de tests no paramétricos)
    * Utilizando como método de control el de rango más pequeño (rangos alineados de Friedman)
 
NOTA: para ANOVA, si se rechazara la hipóstesis nula se podría coger el método de media más grande y hacer tantas comparaciones por pares (t-test) como métodos haya (método de control vs. cada método no control)

Leed el archivo .csv con los resultados de la ejecución de los 4 valores del híper-parámetro k de KNN y realizad el análisis estadístico apropiado para determinar si alguna confirugación es mejor que el resto.

Para ello, cread el código pertinente en la celda posterior (podéis añadir más de una) y también debéis analizar el resultado del test. 
* Para añadir celdas debéis presionar el símbolo *+* de la cinta de opciones. El tipo de celda que viene por defecto es de código (podéis ver *Code* en el menú desplegable que aparece en la opción más a la derecha de la misma cinta de opciones). Para cambiar el tipo de celda a una de tipo texto podéis presionar el desplegable y cambiar de code a *Markdown*.

In [4]:
import pandas as pd

dfKNN = pd.read_csv('resultadosKNN.csv', index_col=0)

dfKNN

                         k1         k3         k5         k7
Metodo                                                      
appendicitis.dat  80.303030  86.839827  88.744589  87.792208
balance.dat       79.040000  81.120000  85.440000  88.160000
bupa.dat          62.608696  63.768116  60.869565  60.000000
ecoli.dat         76.466198  80.654083  81.562774  80.079017
glass.dat         73.322259  70.033223  68.660022  68.183832
haberman.dat      66.340561  69.291380  68.646219  70.253834
iris.dat          94.666667  94.666667  94.666667  94.000000
newthyroid.dat    96.744186  94.883721  93.953488  92.558140
page-blocks.dat   96.089070  96.765270  96.491214  96.399856
pima.dat          69.916815  72.134793  71.611069  71.480350
wine.dat          95.523810  94.412698  96.079365  96.079365
wisconsin.dat     95.756763  96.487763  97.072778  96.926793
yeast.dat         51.551552  53.975794  55.457275  57.143507

In [5]:
from statds.normality import shapiro_wilk_normality

alpha = 0.05

resultadosShapiro = []

for columna in dfKNN.columns:
    resultado = shapiro_wilk_normality(
        dfKNN[columna].to_numpy(),
        alpha
    )
    
    resultadosShapiro.append([
        columna,
        resultado[0],
        resultado[1],
        resultado[2],
        resultado[3]
    ])

dfShapiro = pd.DataFrame(
    resultadosShapiro,
    columns=[
        'Metodo',
        'Estadistico',
        'p-value',
        'Valor critico',
        'Hipotesis'
    ]
)

dfShapiro

  Metodo  Estadistico   p-value Valor critico  \
0     k1     0.902246  0.143566          None   
1     k3     0.902409  0.144306          None   
2     k5     0.885183  0.083853          None   
3     k7     0.883213  0.078836          None   

                                           Hipotesis  
0  Fail to Reject H0 with alpha = 0.05(Same distr...  
1  Fail to Reject H0 with alpha = 0.05(Same distr...  
2  Fail to Reject H0 with alpha = 0.05(Same distr...  
3  Fail to Reject H0 with alpha = 0.05(Same distr...  

In [8]:
dfANOVA = dfKNN.copy()

dfANOVA.insert(0, 'Metodo', dfANOVA.index)

dfANOVA

                            Metodo         k1         k3         k5         k7
Metodo                                                                        
appendicitis.dat  appendicitis.dat  80.303030  86.839827  88.744589  87.792208
balance.dat            balance.dat  79.040000  81.120000  85.440000  88.160000
bupa.dat                  bupa.dat  62.608696  63.768116  60.869565  60.000000
ecoli.dat                ecoli.dat  76.466198  80.654083  81.562774  80.079017
glass.dat                glass.dat  73.322259  70.033223  68.660022  68.183832
haberman.dat          haberman.dat  66.340561  69.291380  68.646219  70.253834
iris.dat                  iris.dat  94.666667  94.666667  94.666667  94.000000
newthyroid.dat      newthyroid.dat  96.744186  94.883721  93.953488  92.558140
page-blocks.dat    page-blocks.dat  96.089070  96.765270  96.491214  96.399856
pima.dat                  pima.dat  69.916815  72.134793  71.611069  71.480350
wine.dat                  wine.dat  95.523810  94.41

In [12]:
resultadoANOVA = anova_cases(dfANOVA, alpha)

print(resultadoANOVA)

(  Groups  Nº Samples       Mean  Std. Dev.  Std. Error
0     k1          13  79.871508  14.415544    3.998152
1     k3          13  81.156410  13.714029    3.803587
2     k5          13  81.481156  14.163814    3.928335
3     k7          13  81.465915  13.858383    3.843624,            Source  Degrees of Freedom (DF)  Sum of Squares (SS)  \
0  Between Groups                        3            22.703067   
1   Within Groups                       48         10251.161750   
2           Total                       51         10273.864817   

   Mean Square (MS)    F-Stat Rejected Value  
0          7.567689  0.035435         2.8387  
1        213.565870                           
2        221.133559                           , 0.03543491803279155, 0.990949260625454, 2.8387, 'Fail to Reject H0 with alpha = 0.05 (Same distributions)')


En primer lugar, se aplicó el test de normalidad de Shapiro-Wilk a los resultados obtenidos para cada configuración de KNN (k=1, k=3, k=5 y k=7). Los resultados no permiten rechazar la hipótesis nula de normalidad para ninguna de las configuraciones, utilizando un nivel de significación de 0.05.
Posteriormente, se aplicó el test de Bartlett para comprobar la homocedasticidad de los resultados. El p-value obtenido fue superior a 0.05, por lo que no se rechaza la hipótesis nula y se consideran homogéneas las varianzas.
Al cumplirse las condiciones de normalidad y homocedasticidad, se aplicó el test paramétrico ANOVA para comparar las cuatro configuraciones de KNN.
 Los resultados obtenidos fueron un estadístico F de 0.035435 y un p-value de 0.990949. Como el p-value es mayor que α=0.05, no se rechaza la hipótesis nula. Por tanto, no existen diferencias  significativas entre las configuraciones k=1, k=3, k=5 y k=7.###

Repetid todo el proceso anterior para comparar el rendimiento de los árboles de decisión generados al modificar el parámetro *max_depth* con los valores 1, 2, 3 y 4. Este parámetro limita la profundidad máxima del árbol de decisión, es decir, el número de niveles máximos del árbol. NOTA: debéis asignar el valor 123 como semilla del árbol de decisión (el mismo valor para la semilala de la validación cruzada).

Todo el resto del proceso es igual al realizado con KNN. Guardad los resultados de todos los datasets y todas las configuraciones del árbol de decisión en una variable llamada  *resultadosEjecucionDT* y generad un archivo llamado *resultadosDT.csv*.

In [13]:
# Se importan las librerías necesarias
from dataset import lecturaDatos
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Valores de max_depth que vamos a probar
MAX_DEPTH = [1, 2, 3, 4]

# Matriz para almacenar los resultados
resultadosEjecucionDT = np.zeros((len(datasets), len(MAX_DEPTH)))

# Bucle para recorrer los diferentes valores de max_depth
for i, max_depth in enumerate(MAX_DEPTH):

    # Lista para almacenar el resultado de cada dataset
    accDatasets = []

    # Bucle para recorrer todos los datasets
    for fichero in datasets:

        # Se leen los datos
        d = lecturaDatos(join("datasets", fichero))

        # Se crea la validación cruzada
        cv = StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=123
        )

        # Lista para almacenar el resultado de cada partición
        accParticiones = []

        # Validación cruzada
        for train_index, test_index in cv.split(d.data, d.target):

            # Se crea la Pipeline
            pipeline = Pipeline([
                ('estandarizacion', StandardScaler()),
                ('clasificador', DecisionTreeClassifier(
                    max_depth=max_depth,
                    random_state=123
                ))
            ])

            # Se entrena el árbol
            pipeline.fit(
                d.data[train_index],
                d.target[train_index]
            )

            # Se predicen las clases del conjunto de test
            predicciones = pipeline.predict(
                d.data[test_index]
            )

            # Se calcula la accuracy
            acc = accuracy_score(
                d.target[test_index],
                predicciones
            ) * 100

            # Se guarda el resultado de la partición
            accParticiones.append(acc)

        # Media de las 5 particiones
        accDatasets.append(np.mean(accParticiones))

    # Se guardan los resultados del max_depth actual
    resultadosEjecucionDT[:, i] = accDatasets


# Crear DataFrame con los resultados
dfDT = pd.DataFrame(
    resultadosEjecucionDT,
    index=datasets,
    columns=['max_depth1', 'max_depth2', 'max_depth3', 'max_depth4']
)

# Nombre de la primera columna
dfDT.index.name = 'Metodo'

# Guardar los resultados en CSV
dfDT.to_csv('resultadosDT.csv')

# Mostrar resultados
dfDT

                  max_depth1  max_depth2  max_depth3  max_depth4
Metodo                                                          
appendicitis.dat   85.930736   86.883117   86.883117   88.787879
balance.dat        58.400000   65.280000   69.600000   78.720000
bupa.dat           60.869565   64.927536   67.246377   65.217391
ecoli.dat          62.199298   72.906058   78.871817   78.270413
glass.dat          46.234773   60.730897   65.880399   63.997785
haberman.dat       70.909572   73.199365   75.489159   72.882073
iris.dat           66.666667   91.333333   93.333333   94.666667
newthyroid.dat     78.139535   88.372093   88.372093   90.697674
page-blocks.dat    93.146928   95.248570   95.888324   96.253721
pima.dat           71.482047   72.914863   73.045582   74.870554
wine.dat           67.412698   82.047619   88.793651   85.984127
wisconsin.dat      91.071275   93.559468   94.292615   94.584586
yeast.dat          40.700928   48.651834   55.998271   57.009055

In [14]:
Test.assertEquals(list(map(lambda x: round(x, 2), resultadosEjecucionDT.mean(axis=0))), [68.7, 76.62, 79.51, 80.15], 'Resultados incorrectos')

1 test passed.


Analizad el resultado del test estadístico anterior. Insertad las celdas que consideréis oportunas.

In [15]:
# Shapiro 
import pandas as pd
from statds.normality import shapiro_wilk_normality

alpha = 0.05

dfDT = pd.read_csv('resultadosDT.csv', index_col=0)

resultadosShapiroDT = []

for columna in dfDT.columns:
    resultado = shapiro_wilk_normality(
        dfDT[columna].to_numpy(),
        alpha
    )

    resultadosShapiroDT.append([
        columna,
        resultado[0],
        resultado[1],
        resultado[2],
        resultado[3]
    ])

dfShapiroDT = pd.DataFrame(
    resultadosShapiroDT,
    columns=[
        'Metodo',
        'Estadistico',
        'p-value',
        'Valor critico',
        'Hipotesis'
    ]
)

dfShapiroDT

       Metodo  Estadistico   p-value Valor critico  \
0  max_depth1     0.966309  0.846691          None   
1  max_depth2     0.944433  0.516799          None   
2  max_depth3     0.938243  0.434552          None   
3  max_depth4     0.935427  0.400470          None   

                                           Hipotesis  
0  Fail to Reject H0 with alpha = 0.05(Same distr...  
1  Fail to Reject H0 with alpha = 0.05(Same distr...  
2  Fail to Reject H0 with alpha = 0.05(Same distr...  
3  Fail to Reject H0 with alpha = 0.05(Same distr...  

Finalmente vamos a comparar la mejor configuración de KNN contra la mejor de los árboles de decisión. En caso de que no haya diferencias estadísticas se debe utilizar el método de control de cada tipo de clasificador.

Para ello, se debe generar un DataFrame con los resultados de los métodos de control tanto de KNN como del árbol de decisión y guardarlo como un fichero .csv. Posteriormente, se debe realizar el estudio estadístico apropiado y analizar los resultados para obtener las conclusiones del estudio. 

Podéis añadir cuantas celdas sean necesarias.

In [17]:
# Crear el Dataframe con los dos métodos de control
# primer paso leer los resultados 
import pandas as pd

dfKNN = pd.read_csv('resultadosKNN.csv', index_col=0)
dfDT = pd.read_csv('resultadosDT.csv', index_col=0)

dfControl = pd.DataFrame({
    'Metodo': dfKNN.index,
    'KNN_k5': dfKNN['k5'].values,
    'DT_depth4': dfDT['max_depth4'].values
})

dfControl

              Metodo     KNN_k5  DT_depth4
0   appendicitis.dat  88.744589  88.787879
1        balance.dat  85.440000  78.720000
2           bupa.dat  60.869565  65.217391
3          ecoli.dat  81.562774  78.270413
4          glass.dat  68.660022  63.997785
5       haberman.dat  68.646219  72.882073
6           iris.dat  94.666667  94.666667
7     newthyroid.dat  93.953488  90.697674
8    page-blocks.dat  96.491214  96.253721
9           pima.dat  71.611069  74.870554
10          wine.dat  96.079365  85.984127
11     wisconsin.dat  97.072778  94.584586
12         yeast.dat  55.457275  57.009055

In [18]:
dfControl.to_csv('resultadosmControl.csv', index=False)

In [19]:
print("Media KNN k=5:", dfControl['KNN_k5'].mean()) # como se puede observar KNN  tiene un valor ligeramente superior pero eso no significa que sea mejor estadísticamente 
print("Media DT max_depth=4:", dfControl['DT_depth4'].mean())

Media KNN k=5: 81.48115582139054
Media DT max_depth=4: 80.14937879958823


In [20]:
# compruebo la normalidad de estos dos métodos Shapiro- Wilk
from statds.normality import shapiro_wilk_normality

alpha = 0.05

resultadosShapiroControl = []

for columna in ['KNN_k5', 'DT_depth4']:

    resultado = shapiro_wilk_normality(
        dfControl[columna].to_numpy(),
        alpha
    )

    resultadosShapiroControl.append([
        columna,
        resultado[0],
        resultado[1],
        resultado[2],
        resultado[3]
    ])

dfShapiroControl = pd.DataFrame(
    resultadosShapiroControl,
    columns=[
        'Metodo',
        'Estadistico',
        'p-value',
        'Valor critico',
        'Hipotesis'
    ]
)

dfShapiroControl

      Metodo  Estadistico   p-value Valor critico  \
0     KNN_k5     0.885183  0.083853          None   
1  DT_depth4     0.935427  0.400470          None   

                                           Hipotesis  
0  Fail to Reject H0 with alpha = 0.05(Same distr...  
1  Fail to Reject H0 with alpha = 0.05(Same distr...  

In [22]:
# Compruebo homocedasticidad
from statds.homoscedasticity import bartlett_test

resultadoBartlettControl = bartlett_test(
    dfControl,
    alpha
)

print("Estadístico:", resultadoBartlettControl[0])
print("p-value:", resultadoBartlettControl[1])
print("Valor crítico:", resultadoBartlettControl[2])
print("Hipótesis:", resultadoBartlettControl[3]) 
# la propia libreria no dice Fail to reject quiere decir que shapiro Wilk tiene nomralidad y Barlett tiene varianzas homogéneas

Estadístico: 0.19930007930443025
p-value: 0.65528639592715
Valor crítico: 3.841
Hipótesis: Fail to Reject H0 with alpha = 0.05 (Same Variances)


In [23]:
from statds.parametrics import t_test_unpaired

resultadoTTest = t_test_unpaired(
    dfControl[['KNN_k5', 'DT_depth4']],
    alpha
)

print("Estadístico:", resultadoTTest[0])
print("p-value:", resultadoTTest[1])
print("Valor crítico:", resultadoTTest[2])
print("Hipótesis:", resultadoTTest[3])

Estadístico: 0.24493525542179287
p-value: 0.8085887192074146
Valor crítico: 1.711
Hipótesis: Fail to Reject H0 with alpha = 0.05 (Same distributions)


Regla:
Si p-value < 0.05 → rechazamos H0 → hay diferencias significativas.
Si p-value ≥ 0.05 → no rechazamos H0 → no hay diferencias significativas.
================================================================================
Por lo tanto como el valor es 0.80 NO se rechaza H0, KNN con k=5 es 81,48% y Árbol , max depth=4 es 80,15; KNN es ligeramente superior en 1.33 puntos pero según el test no puedo afirmar cual es mejor. Por lo tanto, no existen diferencias estadísticamente significativas entre ambos métodos ya que presentan un rendimiento similar en los conjuntos de datos analizados.